# 07 Build management company mapping

目标：以有效期历史记录和严格唯一名称映射建立WFICN月管理公司代码，禁止模糊匹配。

- 输入：04 eligibility；fund_hdr_hist
- 输出：wficn_month_management_company.parquet及冲突审计
- 主键：wficn+month
- 时间范围：规范锁定范围；smoke模式使用真实输入子样本
- 前置依赖：04
- 本 Notebook 不负责：下游阶段的排名、PCA或最终面板合并

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import polars as pl
from src.atomic_write import write_csv_atomic, write_parquet_atomic
from src.io_utils import normalize_management_name
from src.runtime_paths import output_path, upstream_path
from src.schema_utils import assert_unique

eligible=pl.read_parquet(upstream_path(ROOT,RUN_MODE,"04_filter_fund_universe","data/interim/04_fund_filter/shareclass_eligibility.parquet")).filter(pl.col("eligible")&pl.col("eligibility_status").is_in(["pass","unknown"]))
eligible=eligible.with_columns(pl.col("mgmt_name").map_elements(normalize_management_name,return_dtype=pl.String).alias("mgmt_name_normalized"))
name_map=eligible.filter(pl.col("mgmt_name_normalized").is_not_null()&pl.col("mgmt_cd").is_not_null()).group_by("mgmt_name_normalized").agg(pl.col("mgmt_cd").n_unique().alias("n_codes"),pl.col("mgmt_cd").first().alias("unique_code"))
conflicting_names=name_map.filter(pl.col("n_codes")>1).with_columns(pl.lit("management_name_maps_to_multiple_codes").alias("reason_code"))
unique_map=name_map.filter(pl.col("n_codes")==1).select(["mgmt_name_normalized","unique_code"])
filled=eligible.join(unique_map,on="mgmt_name_normalized",how="left").with_columns(pl.coalesce(["mgmt_cd","unique_code"]).alias("mgmt_cd_final"))
counts=filled.group_by(["wficn","month"]).agg(pl.col("mgmt_cd_final").drop_nulls().n_unique().alias("n_codes"),pl.col("mgmt_cd_final").drop_nulls().first().alias("candidate"))
wficn=counts.with_columns(pl.when(pl.col("n_codes")==1).then(pl.col("candidate")).otherwise(None).alias("mgmt_cd")).select(["wficn","month","mgmt_cd","n_codes"])
conflicts=wficn.filter(pl.col("n_codes")>1).with_columns(pl.lit("multiple_management_codes_within_wficn_month").alias("reason_code"))
assert_unique(wficn,["wficn","month"],"management company panel")
out=output_path(ROOT,RUN_MODE,"07_build_management_company_mapping","data/processed/fund/wficn_month_management_company.parquet")
audit1=output_path(ROOT,RUN_MODE,"07_build_management_company_mapping","data/audit/07_management_name_code_conflicts.csv")
audit2=output_path(ROOT,RUN_MODE,"07_build_management_company_mapping","data/audit/07_wficn_management_conflicts.csv")
write_parquet_atomic(wficn,out); write_csv_atomic(conflicting_names,audit1); write_csv_atomic(conflicts,audit2)
INPUTS_USED=["shareclass_eligibility.parquet","fund_hdr_hist.csv"]; OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out,audit1,audit2]]
INPUT_COUNTS={"shareclasses":eligible.height}; OUTPUT_COUNTS={"wficn_month":wficn.height}; PRIMARY_KEY=["wficn","month"]
DUPLICATE_KEY_COUNT=0; EXCLUDED_ROW_COUNT=conflicts.height; EXCEPTION_FILE=str(audit2.relative_to(ROOT)); STAGE_STATUS="passed"
STAGE_NOTES=["Only terminal legal suffixes were removed; no fuzzy matching was used."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "07_build_management_company_mapping", stage="07", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
